# Sentiment analysis of Narrative Segments

This colab file contains the exploration of sentiment analysis in Grimm's tale previously segmented through Text-Tiling.

The author's main contribution is a novel modeling of episode surprisal, which is built on emotions polarity in the segment (calculate_emotion_conflict_surprisal).

The code presented hereunder was mainly AI-generated. The author provided prompting (including selection of methods to be used), revision, and integrations, as well as the data sources for visualizations.

Cosimo Palma

cosimo.palma@protonmail.com

Install required packages

In [ ]:
!pip install transformers torch pandas numpy matplotlib seaborn plotly
!pip install vaderSentiment textblob

Import them

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Sentiment analysis libraries
from transformers import pipeline
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

# Statistical libraries
from scipy import stats
from sklearn.preprocessing import StandardScaler
from collections import defaultdict

def extract_story_title(filename):
    """
    Extract story title from filename format: extract substring after 3rd "_" and before 3rd-last "_"
    Example: "13_grimm_ia1853-1_the_little_brother_and_sister_tt_20250710_110158.txt"
    -> "The Little Brother And Sister"
    """
    # Remove .txt extension
    name_without_ext = filename.replace('.txt', '')

    # Split by underscores
    parts = name_without_ext.split('_')

    if len(parts) < 6:  # Need at least 6 parts to have 3rd and 3rd-last
        return name_without_ext.replace('_', ' ').title()

    # Extract from after 3rd "_" to before 3rd-last "_"
    # parts[3:-3] gives us the story title parts
    story_parts = parts[3:-3]

    # Join with spaces and capitalize each word
    story_title = ' '.join(story_parts).replace('_', ' ').title()

    return story_title

Implementation of the Sentiment Analyzer

In [ ]:
class GrimmSentimentAnalyzer:
    """
    Analyzes sentiment and emotion shifts in Grimm's folktales
    """

    def __init__(self):
        # Initialize sentiment analyzers
        print("Loading sentiment analysis models...")
        self.emotion_analyzer = pipeline("text-classification",
                                        model="j-hartmann/emotion-english-distilroberta-base",
                                        return_all_scores=True)
        self.vader_analyzer = SentimentIntensityAnalyzer()
        self.emotion_labels = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']

    def parse_tale_file(self, file_path):
        """
        Parse a single tale file into segments
        """
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()

        # Split by segment markers
        segments = re.split(r'Segment \d+:', content)

        # Remove empty segments and clean up
        segments = [seg.strip() for seg in segments if seg.strip()]

        # Skip the title (segment 0) and clean segments
        tale_segments = []
        for i, segment in enumerate(segments[1:], 1):  # Skip first segment (title)
            # Remove separator lines
            cleaned_segment = re.sub(r'-{20,}', '', segment).strip()
            if cleaned_segment:
                tale_segments.append({
                    'segment_id': i,
                    'text': cleaned_segment
                })

        return tale_segments

    def analyze_emotions(self, text):
        """
        Analyze emotions in text using multiple methods
        """
        # RoBERTa emotion analysis
        emotion_scores = self.emotion_analyzer(text[:512])  # Truncate for model limits
        emotion_dict = {item['label']: item['score'] for item in emotion_scores[0]}

        # VADER sentiment
        vader_scores = self.vader_analyzer.polarity_scores(text)

        # TextBlob sentiment
        blob = TextBlob(text)
        textblob_polarity = blob.sentiment.polarity
        textblob_subjectivity = blob.sentiment.subjectivity

        return {
            'emotions': emotion_dict,
            'vader': vader_scores,
            'textblob_polarity': textblob_polarity,
            'textblob_subjectivity': textblob_subjectivity
        }

    def calculate_emotion_surprisal(self, emotion_sequence):
        """
        Calculate emotion-based surprisal scores
        """
        surprisal_scores = []

        for i in range(1, len(emotion_sequence)):
            current_emotions = emotion_sequence[i]
            previous_emotions = emotion_sequence[i-1]

            # Calculate emotional distance (KL divergence approximation)
            distance = 0
            for emotion in self.emotion_labels:
                curr_prob = current_emotions.get(emotion, 0.001)  # Small epsilon to avoid log(0)
                prev_prob = previous_emotions.get(emotion, 0.001)
                distance += curr_prob * np.log(curr_prob / prev_prob)

            # Calculate surprisal as negative log probability
            surprisal = -np.log(1 / (1 + distance))
            surprisal_scores.append(surprisal)

        return surprisal_scores

    def calculate_kld_adjacent(self, emotion_sequence, symmetric = False):
        """
        Calculate KL divergence between adjacent segments
        """
        kld_scores = []

        for i in range(1, len(emotion_sequence)):
            current = emotion_sequence[i]
            previous = emotion_sequence[i-1]

            # Convert to numpy arrays and add small epsilon
            curr_probs = np.array([current.get(emotion, 0) for emotion in self.emotion_labels]) + 1e-10
            prev_probs = np.array([previous.get(emotion, 0) for emotion in self.emotion_labels]) + 1e-10

            # Normalize to ensure they sum to 1
            curr_probs = curr_probs / curr_probs.sum()
            prev_probs = prev_probs / prev_probs.sum()
            if symmetric:
              # Calculate symmetric KL divergence (average of both directions)
              kld_forward = np.sum(curr_probs * np.log(curr_probs / prev_probs))
              kld_backward = np.sum(prev_probs * np.log(prev_probs / curr_probs))
              symmetric_kld = (kld_forward + kld_backward) / 2
              kld_scores.append(symmetric_kld)
            else:
              # Calculate regular KL divergence
              kld_forward = np.sum(curr_probs * np.log(curr_probs / prev_probs))
              asymmetric_kld = kld_forward
              kld_scores.append(asymmetric_kld)

        return kld_scores

    def calculate_kld_contextual(self, emotion_sequence, window_size=3, symmetric = False):
        """
        Calculate KL divergence using contextual windows
        """
        kld_scores = []

        for i in range(len(emotion_sequence)):
            if i < window_size:
                kld_scores.append(0.0)  # Not enough context for early segments
                continue

            # Current segment
            current = emotion_sequence[i]
            curr_probs = np.array([current.get(emotion, 0) for emotion in self.emotion_labels]) + 1e-10
            curr_probs = curr_probs / curr_probs.sum()

            # Context window (average of previous segments)
            context_emotions = {emotion: [] for emotion in self.emotion_labels}
            for j in range(max(0, i-window_size), i):
                for emotion in self.emotion_labels:
                    context_emotions[emotion].append(emotion_sequence[j].get(emotion, 0))

            context_probs = np.array([np.mean(context_emotions[emotion]) for emotion in self.emotion_labels]) + 1e-10
            context_probs = context_probs / context_probs.sum()

            if symmetric:
              # Calculate symmetric KL divergence (average of both directions)
              kld_forward = np.sum(curr_probs * np.log(curr_probs / context_probs))
              kld_backward = np.sum(context_probs * np.log(context_probs / curr_probs))
              symmetric_kld = (kld_forward + kld_backward) / 2
              kld_scores.append(symmetric_kld)
            else:
              # Calculate regular KL divergence
              kld_forward = np.sum(curr_probs * np.log(curr_probs / context_probs))
              asymmetric_kld = kld_forward
              kld_scores.append(asymmetric_kld)

        return kld_scores

    def calculate_emotion_conflict_surprisal(self, emotion_sequence):
        """
        Calculate emotion surprisal based on rhomboid area formed by conflicting emotions
        High surprisal occurs when opposite emotions are both high (emotional conflict)
        Normalized to range 0-2
        """
        surprisal_scores = []

        for emotions in emotion_sequence:
            # Extract the 4 key emotions
            sadness = emotions.get('sadness', 0)
            joy = emotions.get('joy', 0)
            fear = emotions.get('fear', 0)
            anger = emotions.get('anger', 0)

            normalized_area = (joy + sadness) * (fear + anger) * 25

            surprisal_scores.append(normalized_area)

        return surprisal_scores

    def analyze_tale(self, file_path):
        """
        Analyze a single tale file
        """
        tale_name = extract_story_title(file_path)
        segments = self.parse_tale_file(file_path)

        if not segments:
            print(f"No segments found in {tale_name}")
            return None

        print(f"Analyzing {tale_name} with {len(segments)} segments...")

        results = []
        emotion_sequence = []

        for segment in segments:
            analysis = self.analyze_emotions(segment['text'])

            result = {
                'tale': tale_name,
                'segment_id': segment['segment_id'],
                'text': segment['text'],
                'text_length': len(segment['text']),
                **analysis['emotions'],
                'vader_compound': analysis['vader']['compound'],
                'vader_pos': analysis['vader']['pos'],
                'vader_neu': analysis['vader']['neu'],
                'vader_neg': analysis['vader']['neg'],
                'textblob_polarity': analysis['textblob_polarity'],
                'textblob_subjectivity': analysis['textblob_subjectivity']
            }

            results.append(result)
            emotion_sequence.append(analysis['emotions'])

        # Calculate surprisal scores
        surprisal_scores = self.calculate_emotion_surprisal(emotion_sequence)

        # Add surprisal scores to results (starting from segment 2)
        for i, score in enumerate(surprisal_scores):
            results[i+1]['emotion_surprisal'] = score

        # First segment has no surprisal (no previous segment to compare)
        results[0]['emotion_surprisal'] = 0.0

        return pd.DataFrame(results)

    def analyze_tale_with_methods(self, file_path, segmentation_method="text_tiling", comparison_type="contextual"):
        """
        Analyze a tale with specific segmentation and comparison methods
        """
        tale_name = extract_story_title(file_path)
        # For now, we'll use the existing segmentation (would need to implement text_tiling vs equal_length)
        segments = self.parse_tale_file(file_path)

        if not segments:
            return None

        # Analyze emotions for each segment
        emotion_sequence = []
        results = []

        for segment in segments:
            analysis = self.analyze_emotions(segment['text'])
            emotion_sequence.append(analysis['emotions'])

            result = {
                'tale': tale_name,
                'segment_id': segment['segment_id'],
                'text': segment['text'],
                'text_length': len(segment['text']),
                'tokens': len(segment['text'].split()),  # Simple token count
                **analysis['emotions'],
                'vader_compound': analysis['vader']['compound'],
                'vader_pos': analysis['vader']['pos'],
                'vader_neu': analysis['vader']['neu'],
                'vader_neg': analysis['vader']['neg'],
                'textblob_polarity': analysis['textblob_polarity'],
                'textblob_subjectivity': analysis['textblob_subjectivity']
            }
            results.append(result)

        # Calculate KL divergence based on comparison type
        if comparison_type == "adjacent":
            kld_scores = self.calculate_kld_adjacent(emotion_sequence)
        else:  # contextual
            kld_scores = self.calculate_kld_contextual(emotion_sequence)

        # Add KLD scores to results (pad with 0 for first segment(s))
        for i, score in enumerate(kld_scores):
            if comparison_type == "adjacent":
                results[i+1]['kld_score'] = score
            else:  # contextual - already handles padding
                results[i]['kld_score'] = score

        # Set first segment(s) KLD to 0
        if comparison_type == "adjacent":
            results[0]['kld_score'] = 0.0

        return pd.DataFrame(results), kld_scores

    def analyze_folder(self, folder_path):
        """
        Analyze all tale files in a folder
        """
        all_results = []

        for filename in os.listdir(folder_path):
            if filename.endswith('.txt'):
                file_path = os.path.join(folder_path, filename)
                tale_df = self.analyze_tale(file_path)
                if tale_df is not None:
                    all_results.append(tale_df)

        if not all_results:
            print("No valid tale files found!")
            return None

        return pd.concat(all_results, ignore_index=True)

    def visualize_emotion_flow(self, df, tale_name):
        """
        Create visualization of emotion flow for a specific tale
        """
        tale_data = df[df['tale'] == tale_name].sort_values('segment_id')

        if tale_data.empty:
            print(f"No data found for tale: {tale_name}")
            return

        # Calculate emotion conflict surprisal if not already present
        if 'emotion_conflict_surprisal' not in tale_data.columns:
            emotion_sequence = []
            for _, row in tale_data.iterrows():
                emotions = {emotion: row[emotion] for emotion in self.emotion_labels}
                emotion_sequence.append(emotions)

            conflict_scores = self.calculate_emotion_conflict_surprisal(emotion_sequence)
            tale_data = tale_data.copy()
            tale_data['emotion_conflict_surprisal'] = conflict_scores

        fig = make_subplots(
            rows=3, cols=1,
            subplot_titles=('Emotion Distribution', 'Sentiment Over Time', 'Emotion Surprisal'),
            vertical_spacing=0.1
        )

        # Plot 1: Emotion distribution
        segments = tale_data['segment_id'].values
        for emotion in self.emotion_labels:
            fig.add_trace(
                go.Scatter(x=segments, y=tale_data[emotion].values,
                          mode='lines+markers', name=emotion),
                row=1, col=1
            )

        # Plot 2: Sentiment scores
        fig.add_trace(
            go.Scatter(x=segments, y=tale_data['vader_compound'].values,
                      mode='lines+markers', name='VADER Compound', line=dict(color='blue')),
            row=2, col=1
        )
        fig.add_trace(
            go.Scatter(x=segments, y=tale_data['textblob_polarity'].values,
                      mode='lines+markers', name='TextBlob Polarity', line=dict(color='red')),
            row=2, col=1
        )

        # Plot 3: Both emotion surprisal measures
        if 'emotion_surprisal' in tale_data.columns:
            fig.add_trace(
                go.Scatter(x=segments, y=tale_data['emotion_surprisal'].values,
                          mode='lines+markers', name='KL Divergence Surprisal', line=dict(color='purple')),
                row=3, col=1
            )

        fig.add_trace(
            go.Scatter(x=segments, y=tale_data['emotion_conflict_surprisal'].values,
                      mode='lines+markers', name='Emotion Conflict Surprisal', line=dict(color='orange')),
            row=3, col=1
        )

        fig.update_layout(height=800, title_text=f"Emotion Analysis: {tale_name}")
        fig.update_xaxes(title_text="Segment ID", row=3, col=1)
        fig.update_yaxes(title_text="Emotion Score", row=1, col=1)
        fig.update_yaxes(title_text="Sentiment Score", row=2, col=1)
        fig.update_yaxes(title_text="Surprisal Score", row=3, col=1)

        fig.show()

    def create_summary_statistics(self, df):
        """
        Create summary statistics for all tales
        """
        summary_stats = {}

        # Overall statistics
        summary_stats['total_tales'] = df['tale'].nunique()
        summary_stats['total_segments'] = len(df)
        summary_stats['avg_segments_per_tale'] = df.groupby('tale')['segment_id'].count().mean()

        # Emotion statistics
        emotion_means = df[self.emotion_labels].mean()
        summary_stats['dominant_emotion'] = emotion_means.idxmax()
        summary_stats['emotion_distribution'] = emotion_means.to_dict()

        # Surprisal statistics
        summary_stats['avg_surprisal'] = df['emotion_surprisal'].mean()
        summary_stats['max_surprisal'] = df['emotion_surprisal'].max()
        summary_stats['surprisal_std'] = df['emotion_surprisal'].std()

        # Most surprising segments
        most_surprising = df.nlargest(5, 'emotion_surprisal')[['tale', 'segment_id', 'emotion_surprisal']]
        summary_stats['most_surprising_segments'] = most_surprising.to_dict('records')

        return summary_stats

    def plot_surprisal_heatmap(self, df):
        """
        Create heatmap of emotion surprisal across all tales
        """
        # Create pivot table for heatmap
        pivot_data = df.pivot_table(
            values='emotion_surprisal',
            index='tale',
            columns='segment_id',
            aggfunc='mean'
        )

        plt.figure(figsize=(15, 8))
        sns.heatmap(pivot_data, annot=False, cmap='YlOrRd', cbar_kws={'label': 'Emotion Surprisal'})
        plt.title('Emotion Surprisal Heatmap Across All Tales')
        plt.xlabel('Segment ID')
        plt.ylabel('Tale')
        plt.tight_layout()
        plt.show()

    def create_csv_summary(self, folder_path, output_file="sentiment_analysis_summary.csv",
                      segmentation_method="text_tiling", comparison_type="adjacent"):
        """
        Create CSV summary file with statistics for all tales
        """
        all_results = []

        for filename in os.listdir(folder_path):
            if filename.endswith('.txt'):
                file_path = os.path.join(folder_path, filename)

                try:
                    tale_df, kld_scores = self.analyze_tale_with_methods(
                        file_path, segmentation_method, comparison_type
                    )

                    if tale_df is not None and len(kld_scores) > 0:
                        # Calculate summary statistics
                        tale_name = tale_df['tale'].iloc[0]
                        total_chars = tale_df['text_length'].sum()
                        total_tokens = tale_df['tokens'].sum()
                        total_segments = len(tale_df)

                        # Average emotion scores
                        emotion_avgs = {}
                        for emotion in self.emotion_labels:
                            emotion_avgs[f'avg_{emotion}'] = tale_df[emotion].mean()

                        # Average sentiment scores
                        avg_vader = tale_df['vader_compound'].mean()
                        avg_textblob = tale_df['textblob_polarity'].mean()
                        avg_subjectivity = tale_df['textblob_subjectivity'].mean()

                        summary_row = {
                            'ID': filename[:2],
                            'File': filename,
                            'Story':  self.extract_story_title(filename),
                            'Characters': total_chars,
                            'Tokens': total_tokens,
                            'Segments': total_segments,
                            'Avg_Vader': avg_vader,
                            'Avg_TextBlob': avg_textblob,
                            'Avg_Subjectivity': avg_subjectivity,
                            **emotion_avgs,
                            'Segmentation_Method': segmentation_method,
                            'Comparison_Type': comparison_type
                        }

                        all_results.append(summary_row)

                except Exception as e:
                    print(f"Error processing {filename}: {e}")
                    continue

        if all_results:
            summary_df = pd.DataFrame(all_results)
            summary_df.to_csv(output_file, index=False)
            print(f"Summary CSV saved to: {output_file}")
            return summary_df
        else:
            print("No valid results to save.")
            return None


Initialization and usage

In [ ]:
analyzer = GrimmSentimentAnalyzer()

# Example usage:
# 1. Analyze a single tale file
df_single = analyzer.analyze_tale('/content/data/04_grimm_pg19068_little_red-cap_tt_20250710_110157.txt')

# 2. Analyze all tales in a folder
df_all = analyzer.analyze_folder('/content/data')

# 3. Visualize emotion flow for a specific tale
#analyzer.visualize_emotion_flow(df_all, 'rumpelstiltskin')

# 4. Create summary statistics
summary = analyzer.create_summary_statistics(df_all)
# print(summary)

# 5. Plot surprisal heatmap
# analyzer.plot_surprisal_heatmap(df_all)

# Sample function to get started quickly
def quick_analysis(folder_path):
    """
    Quick analysis function for immediate results
    """
    print("Starting analysis of Grimm's folktales...")

    # Analyze all tales
    df = analyzer.analyze_folder(folder_path)

    if df is None:
        print("No tales found to analyze!")
        return

    print(f"Analysis complete! Processed {df['tale'].nunique()} tales with {len(df)} segments total.")

    # Display sample results
    print("\nSample results:")
    print(df[['tale', 'segment_id', 'joy', 'sadness', 'fear', 'anger', 'emotion_surprisal']].head(10))

    # Show summary statistics
    summary = analyzer.create_summary_statistics(df)
    print(f"\nSummary Statistics:")
    print(f"Average surprisal score: {summary['avg_surprisal']:.3f}")
    print(f"Maximum surprisal score: {summary['max_surprisal']:.3f}")
    print(f"Dominant emotion: {summary['dominant_emotion']}")

    # Plot surprisal heatmap
    analyzer.plot_surprisal_heatmap(df)

    # Show most surprising segments
    print("\nMost emotionally surprising segments:")
    for segment in summary['most_surprising_segments']:
        print(f"- {segment['tale']}, Segment {segment['segment_id']}: {segment['emotion_surprisal']:.3f}")

    return df

# To use this notebook:
# 1. Upload your folder containing the tale files
# 2. Run: df = quick_analysis('path/to/your/folder')
# 3. Explore individual tales: analyzer.visualize_emotion_flow(df, 'tale_name')

print("Grimm's Folktales Sentiment Analyzer is ready!")


In [ ]:
df = quick_analysis('/content/data')

In [ ]:
analyzer.visualize_emotion_flow(df, 'Snow-White')

In [ ]:
def quick_csv_analysis(folder_path, output_base="grimm_sentiment_analysis"):
    """
    Quick analysis function that creates both summary and detailed CSVs
    """
    print("Starting CSV analysis of Grimm's folktales...")
    comparison_type="contextual"
    # Create summary CSV (like your example format)
    summary_df = analyzer.create_csv_summary(
        folder_path,
        f"{output_base}_summary_{comparison_type}.csv", #_{comparison_type}
        segmentation_method="text_tiling",
        comparison_type = comparison_type
    )

    if summary_df is not None:
        print(f"\nSummary: Processed {len(summary_df)} tales")
        print("\nSample summary data:")
        print(summary_df[['Story', 'Characters', 'Tokens', 'Segments']].head())

    return summary_df

# Usage:
summary_df = quick_csv_analysis('/content/AI_data/tt_start')